In [21]:
import duckdb
from pathlib import Path

# 1. Establish Directory Paths
BASE_DIR = Path.cwd().parent
RESULT_DIR = BASE_DIR / "results"

# Connect to an in-memory DuckDB database (no files saved to disk)
con = duckdb.connect(database=":memory:")

print("=== LOADING CLEANED DATASETS INTO IN-MEMORY DUCKDB ===")

# 2. Load Dimension & Fact Tables from Saved CSVs
con.execute(f"""
    CREATE OR REPLACE TABLE dim_train AS 
    SELECT * FROM read_csv_auto('{RESULT_DIR / "dim_train_resolved.csv"}');
""")
print("  -> Loaded dim_train (Train Dimension)")

con.execute(f"""
    CREATE OR REPLACE TABLE dim_station AS 
    SELECT * FROM read_csv_auto('{RESULT_DIR / "dim_station_resolved.csv"}');
""")
print("  -> Loaded dim_station (Station Dimension)")

con.execute(f"""
    CREATE OR REPLACE TABLE dim_date AS 
    SELECT * FROM read_csv_auto('{RESULT_DIR / "dim_date.csv"}');
""")
print("  -> Loaded dim_date (Time Dimension)")

con.execute(f"""
    CREATE OR REPLACE TABLE schedule AS 
    SELECT * FROM read_csv_auto('{RESULT_DIR / "combined_schedule_imputed.csv"}');
""")
print("  -> Loaded schedule (Schedule Fact)")

con.execute(f"""
    CREATE OR REPLACE TABLE delay AS 
    SELECT * FROM read_csv_auto('{RESULT_DIR / "combined_delay_cleaned.csv"}');
""")
print("  -> Loaded delay (Delay Fact)")

# ==============================================================================
# 3. DAY CONSISTENCY VERIFICATION QUERY
# ==============================================================================
print("\n=== RUNNING DAY CONSISTENCY CHECK ===")

day_consistency_query = """
WITH JoinedOperations AS (
    SELECT 
        d.date AS origin_date, 
        d.train_no, 
        d.station_no, 
        d.delay, 
        s.arrival_day, 
        s.arrival_time,
        
        -- Calculate actual arrival timestamp by adding days, time components, and delay minutes
        (
            CAST(d.date AS TIMESTAMP) 
            + INTERVAL (s.arrival_day - 1) DAY 
            + (EXTRACT(HOUR FROM s.arrival_time) * INTERVAL 1 HOUR)
            + (EXTRACT(MINUTE FROM s.arrival_time) * INTERVAL 1 MINUTE)
            + (EXTRACT(SECOND FROM s.arrival_time) * INTERVAL 1 SECOND)
            + INTERVAL (d.delay) MINUTE
        ) AS actual_arrival_timestamp,
        
        -- Baseline scheduled arrival timestamp without delay
        (
            CAST(d.date AS TIMESTAMP) 
            + INTERVAL (s.arrival_day - 1) DAY 
            + (EXTRACT(HOUR FROM s.arrival_time) * INTERVAL 1 HOUR)
            + (EXTRACT(MINUTE FROM s.arrival_time) * INTERVAL 1 MINUTE)
            + (EXTRACT(SECOND FROM s.arrival_time) * INTERVAL 1 SECOND)
        ) AS scheduled_arrival_timestamp
        
    FROM delay d 
    JOIN schedule s ON d.train_no = s.train_no AND d.station_no = s.station_no 
    WHERE s.arrival_time IS NOT NULL AND d.delay IS NOT NULL
)
SELECT 
    COUNT(*) AS total_records_checked,
    COUNT(CASE WHEN CAST(actual_arrival_timestamp AS DATE) = CAST(scheduled_arrival_timestamp AS DATE) THEN 1 END) AS same_day_arrivals,
    COUNT(CASE WHEN CAST(actual_arrival_timestamp AS DATE) <> CAST(scheduled_arrival_timestamp AS DATE) THEN 1 END) AS date_shifted_arrivals
FROM JoinedOperations;
"""

results = con.execute(day_consistency_query).df()
print(results.to_string(index=False))

con.close()
print("\n-> Check complete. No database file was created on disk.")

=== LOADING CLEANED DATASETS INTO IN-MEMORY DUCKDB ===
  -> Loaded dim_train (Train Dimension)
  -> Loaded dim_station (Station Dimension)
  -> Loaded dim_date (Time Dimension)
  -> Loaded schedule (Schedule Fact)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  -> Loaded delay (Delay Fact)

=== RUNNING DAY CONSISTENCY CHECK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_records_checked  same_day_arrivals  date_shifted_arrivals
              36623269           35889871                 733398

-> Check complete. No database file was created on disk.
